# Chapter 05 — Sessions and Where They Live

**Companion to *Pi Agents*** · [`Pi Agents` chapter 05](https://programmer.ie/books/pi/05-chapter/)

| | |
|---|---|
| Chapter | `05-chapter.md` · weight 5 · `/books/pi/05-chapter/` |
| Notebook | `notebooks/pi/05-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

For one real session file: **which lines are on disk, and which of them ever
reach the model?**

"Stored" and "sent" are two different sets. The chapter's argument is that a
session file is a complete record, that the request is a projection of it, and
that a tool which confuses the two will quietly lose data.


## What this notebook establishes

- The file is JSONL: a header, then one entry per message, in order, each with an `id` and a `parentId`. For a linear conversation each parent is the entry before it.
- **Stored is not sent.** `model_change`, `thinking_level_change` and an extension's own `custom` entry are on disk and appear nowhere in the next provider request.
- A `context_edit` with no replacement **hides** an earlier message from future requests without deleting it from the file — and the effect is **branch-relative**: navigating to a point before the edit brings the message back.
- Session location precedence is `--session-dir` > `PI_CODING_AGENT_SESSION_DIR` > the `sessionDir` setting, and the default group directory name is the working directory flattened.

## What this notebook does **not** establish

- **The file format has no stability guarantee** for a third-party consumer, and the header `version` has already changed twice across releases. What is shown here is what the pinned release writes.
- **Entry counts are small on purpose.** Three turns is enough to show the shape; nothing here measures a long session.
- **Nothing about how a provider maps these messages onto its wire format** — that is chapter 36 and it is a different question.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — a real persisted session written to a real file, then read back.
* **Evidence scope:** `in_process_runtime` for the history; the shipped binary for the directory precedence. The provider is scripted.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(5))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/05-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: one real session file, read line by line

`harness/history.ts` builds the same three-turn conversation the book's chapters
share — question A, a question B that calls `bash`, question C — and persists it to
a real JSONL file through `SessionManager.create`.

The file is then read directly. Nothing is interpreted: this is the bytes.

In [3]:
DRIVER = r'''
import { readFileSync } from "node:fs";
import { buildHistory, requestFor, userTexts } from "../harness/history.ts";

const rows = (file: string) => readFileSync(file, 'utf8').trim().split('\n').map((l) => JSON.parse(l));

const { h, file, entries, userEntry } = await buildHistory();
const before = entries().map((e) => e.type);

// A `custom` entry written by an extension: Pi's own record of something that
// happened, with no message and therefore no place in the model's context.
h.session.sessionManager.appendCustomEntry('my-ext.counter', { count: 42 });

const sent = await requestFor(h, 'next question');
const disk = rows(file);
h.dispose();

console.log(JSON.stringify({
  headerType: disk[0].type,
  headerHasParentId: Object.prototype.hasOwnProperty.call(disk[0], 'parentId'),
  diskEntryTypes: disk.map((e) => e.type),
  diskMessageRoles: disk.filter((e) => e.type === 'message').map((e) => e.message.role),
  firstEntryIsRoot: disk[1]?.parentId === null,
  parentIsPreviousEntry: disk.slice(2).every((e, i) => e.parentId === disk[i + 1].id),
  sentRoles: sent.map((m) => m.role),
  sentUserTexts: userTexts(sent),
  sentMentionsCustomType: JSON.stringify(sent).includes('my-ext.counter'),
  sentMentionsModelChange: /model_change|thinking_level_change/.test(JSON.stringify(sent)),
  entryTypesBeforeExtensionEntry: before,
  entryTypesNow: disk.map((e) => e.type),
}));
'''
disk = pinb.driver(DRIVER, label="ch05-stored-vs-sent", timeout=240)
print(pinb.table(["", "value"],
                 [[k, v] for k, v in disk.items() if not isinstance(v, list)], indent=""))
print()
print("entry types on disk, in order:")
for i, t in enumerate(disk["diskEntryTypes"], 1):
    print(f"  {i:2d}. {t}")

  value
-----------------------  -------
headerType  session
headerHasParentId  False
firstEntryIsRoot  True
parentIsPreviousEntry  True
sentMentionsCustomType  False
sentMentionsModelChange  False

entry types on disk, in order:
   1. session
   2. model_change
   3. thinking_level_change
   4. message
   5. message
   6. message
   7. message
   8. message
   9. message
  10. message
  11. message
  12. message
  13. custom
  14. message
  15. message


Three things to notice before the assertions.

* `diskEntryTypes` contains `session`, then `message` entries, then at least one
  `model_change` or `thinking_level_change`, then `custom`.
* `sentRoles` contains **no trace** of the custom entry. It is on disk and gone
  from the request.
* `parentIsPreviousEntry` holds once past the root: for a linear conversation the
  tree degenerates to a list, and the file is still a tree. The first entry after the
  header roots it at `parentId: null`.

In [4]:
pinb.show_checks([
    pinb.check("the first line is the session header", disk["headerType"] == "session", disk["headerType"]),
    pinb.check("the header is not part of the tree", not disk["headerHasParentId"], "no parentId on the header"),
    pinb.check("every entry carries the role it stores",
               disk["diskMessageRoles"][:4] == ["system", "user", "assistant", "user"],
               str(disk["diskMessageRoles"][:4])),
    pinb.check("a tool result is stored like any other message",
               "toolResult" in disk["diskMessageRoles"], "present"),
    pinb.check("the first entry roots the tree at null", disk["firstEntryIsRoot"], "parentId: null"),
    pinb.check("after the root, each parent is the entry before it",
               disk["parentIsPreviousEntry"], "chain holds over the rest"),
    pinb.check("the file contains an entry type the request does not",
               "custom" in disk["diskEntryTypes"] and "custom" not in [t for t in disk["sentRoles"]],
               "custom on disk, absent from the request"),
    pinb.check("model and thinking-level entries never reach the provider",
               not disk["sentMentionsModelChange"], "absent"),
    pinb.check("the extension's custom entry never reaches the provider",
               not disk["sentMentionsCustomType"], "absent"),
    pinb.check("the earlier user messages do reach the provider",
               len(disk["sentUserTexts"]) >= 3, str(len(disk["sentUserTexts"])) + " user messages"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------------------  ---------------------------------------
PASS  the first line is the session header  session
PASS  the header is not part of the tree  no parentId on the header
PASS  every entry carries the role it stores  ['system', 'user', 'assistant', 'user']
PASS  a tool result is stored like any other message  present
PASS  the first entry roots the tree at null  parentId: null
PASS  after the root, each parent is the entry before it  chain holds over the rest
PASS  the file contains an entry type the request does not  custom on disk, absent from the request
PASS  model and thinking-level entries never reach the provider  absent
PASS  the extension's custom entry never reaches the provider  absent
PASS  the earlier user messages do reach the provider  4 user messages

10/10 assertions held.


## Experiment: a `context_edit` that hides without deleting

A `context_edit` entry with no replacement removes an earlier message from future
requests. The file still has it. And because the file is a tree, the removal
belongs to a *branch*: go back to before the edit and the message is sent again.

That last part is the one worth seeing, because it is the difference between
"deleted" and "not on this path".

In [5]:
DRIVER2 = r'''
import { readFileSync } from "node:fs";
import { buildHistory, requestFor, userTexts } from "../harness/history.ts";

const rows = (file: string) => readFileSync(file, 'utf8').trim().split('\n').map((l) => JSON.parse(l));
const out: any = {};

// (1) replacement form: the model sees new text, the file keeps the original.
{
	const { h, file, userEntry } = await buildHistory();
	const target = userEntry('question A');
	h.session.sessionManager.appendContextEdit(target.id, { content: 'REDACTED' });
	const sent = await requestFor(h, 'probe');
	out.replacement = {
		fileStillHasOriginal: userTexts(rows(file).filter((e) => e.type === 'message').map((e) => e.message)).includes('question A'),
		sentTexts: userTexts(sent),
		fileHasContextEdit: rows(file).some((e) => e.type === 'context_edit'),
	};
	h.dispose();
}

// (2) removal form, then navigate to a point BEFORE the edit.
{
	const { h, file, userEntry } = await buildHistory();
	const target = userEntry('question A');
	h.session.sessionManager.appendContextEdit(target.id, null);
	const hidden = await requestFor(h, 'probe 1');
	const answerToA = rows(file).find((e) => e.type === 'message' && e.parentId === target.id && e.message.role === 'assistant');
	await h.session.navigateTree(answerToA.id, { summarize: false } as any);
	const restored = await requestFor(h, 'probe 2');
	out.removal = {
		hiddenAfterEdit: userTexts(hidden).includes('question A'),
		stillInFile: rows(file).some((e) => e.type === 'message' && JSON.stringify(e).includes('question A')),
		restoredAfterNavigatingBack: userTexts(restored).includes('question A'),
		leafBeforeEdit: answerToA.id,
	};
	h.dispose();
}

console.log(JSON.stringify(out));
'''
edits = pinb.driver(DRIVER2, label="ch05-context-edit", timeout=300)
print(pinb.md_table(
    ["context_edit form", "in the file", "sent to the model"],
    [["replacement: { content: REDACTED }",
      f"original preserved: {edits['replacement']['fileStillHasOriginal']}",
      f"question A present: {'question A' in edits['replacement']['sentTexts']}"],
     ["removal: null",
      f"original preserved: {edits['removal']['stillInFile']}",
      f"after the edit: {edits['removal']['hiddenAfterEdit']}; after navigating back: {edits['removal']['restoredAfterNavigatingBack']}"]],
))

| context_edit form | in the file | sent to the model |
|---|---|---|
| replacement: { content: REDACTED } | original preserved: True | question A present: False |
| removal: null | original preserved: True | after the edit: False; after navigating back: True |


In [6]:
r, m = edits["replacement"], edits["removal"]

pinb.show_checks([
    pinb.check("a replacement edit leaves the file's own text untouched",
               r["fileStillHasOriginal"], "the stored entry still says question A"),
    pinb.check("a replacement edit writes a context_edit entry",
               r["fileHasContextEdit"], "recorded"),
    pinb.check("a removal edit hides the message from the next request",
               not m["hiddenAfterEdit"], "question A absent"),
    pinb.check("the hidden message is still in the file",
               m["stillInFile"], "nothing deleted"),
    pinb.check("navigating to a point before the edit brings it back",
               m["restoredAfterNavigatingBack"], "the edit is branch-relative"),
])

== Assertions ==
  assertion  observed
----  -------------------------------------------------------  --------------------------------------
PASS  a replacement edit leaves the file's own text untouched  the stored entry still says question A
PASS  a replacement edit writes a context_edit entry  recorded
PASS  a removal edit hides the message from the next request  question A absent
PASS  the hidden message is still in the file  nothing deleted
PASS  navigating to a point before the edit brings it back  the edit is branch-relative

5/5 assertions held.


## Contrasting case: where the file goes

A session's location is a three-level precedence, and it is worth checking because
it is the one configuration Pi reads **before** trust is resolved (chapter 8).

The canonical evidence for all of this — including the three-way precedence — is
the chapter's own test file, which drives the shipped binary for the location
rows.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = 'ch05-disk/disk.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch05-disk/disk.test.ts', show="session")

$ node --test ch05-disk/disk.test.ts
  PASS  8 passed, 0 failed, 0 skipped  in 2707 ms

  tests matching 'session':
    ok   the default session location is --<path>--, with / \\ and : replaced by - and the leading separator removed
    ok   session directory precedence: --session-dir beats PI_CODING_AGENT_SESSION_DIR, which beats the sessionDir setting
    ok   deleting a session is deleting its file: nothing else holds it


## Your turn: predict, then run

**Predict first.** Write a `context_edit` for question **B** instead of question A,
navigate back before it, and ask whether question A is still sent.

**Then change one input.** `HIDE` below selects which user message is hidden.
Run it with `"A"` and with `"B"`.

**Predict the boundary.** What is the difference between `appendContextEdit(id,
null)` and `appendContextEdit(id, { content: "" })`? One removes the message; the
other sends the model an empty message. Which is safer for a secret, and why?

In [8]:
HIDE = "B"

EXERCISE = DRIVER2.replace(
    "const target = userEntry('question A');",
    "const target = userEntry(`question ${process.env.NB_HIDE}`);",
)
print("hiding the entry for question", HIDE)
out = pinb.driver(EXERCISE, label="ch05-hide", timeout=300, env={"NB_HIDE": HIDE})
print()
for form, data in out.items():
    print(f"  {form:12s} {data}")

hiding the entry for question B



  replacement  {'fileStillHasOriginal': True, 'sentTexts': ['question A', 'REDACTED', 'question C', 'probe'], 'fileHasContextEdit': True}
  removal      {'hiddenAfterEdit': True, 'stillInFile': True, 'restoredAfterNavigatingBack': True, 'leafBeforeEdit': 'cf84ae35'}


## Interpretation

The chapter's practical claim is that **the file is the record and the request is
a projection of it**, and the three experiments above are three consequences:

1. **You can store facts the model must not see.** `model_change`,
   `thinking_level_change` and an extension's `custom` entries are all durable,
   branch-relative and invisible. Chapter 19 is built entirely on that.
2. **Editing for the model and editing the record are different operations.** A
   `context_edit` with a replacement rewrites what is sent; the stored entry keeps
   its original text. If you are building an audit trail, the file is the audit
   trail.
3. **Redaction is a policy decision, not a deletion.** `appendContextEdit(id, null)`
   hides a message from this branch. It does not remove it from the file, it does
   not survive navigating back, and it does not touch any other branch. For a
   genuinely secret value, none of that is enough — which is chapter 42's subject.

### What this chapter does not settle

`metadata/05-chapter.yaml` records that there is no stable public schema guarantee
for a third-party consumer, and that the header `version` moves between releases.
The file is readable, not contract.

## Sources, execution mode and limitations

**Execution mode:** **Run** — three real persisted sessions written to real JSONL files and read back; plus the chapter's own test file, which drives the shipped binary for the location rows.

**Evidence scope:** `in_process_runtime` plus `shipped_binary` (the directory-precedence rows).

### What was read or run

- **Ran** `buildHistory()` three times — once for the stored-vs-sent diff, once for the replacement edit, once for the removal-plus-navigation case — each with its own temporary agent directory.
- **Read** the JSONL files directly with `readFileSync`; no interpretation layer sits between the assertion and the bytes.
- **Ran** `ch05-disk/disk.test.ts`.
- **Read** `examples/evidence.json`, chapter 5 rows (8 claims).

### Limitations

- **Three turns.** Enough to show every entry type the chapter names, not enough to say anything about a long session's growth or about compaction interacting with these entries (chapter 23).
- **The format is readable, not guaranteed.** The header `version` is 3 at this pin and has changed twice; `metadata/05-chapter.yaml` records that there is no stability promise for third-party readers.
- **Nothing about a session being portable between machines or between Pi installs.** Deleting a session is deleting its file, and nothing else holds it — which the chapter's own tests assert and this notebook does not repeat.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
